Fundamentals of Deep Learning Models

# Lab 08-1: Sequence Model of RNN
## Exercise: Character-based text generation using a vanilla RNN

This exercise implements a **vanilla RNN** (Section 8.1) from scratch using NumPy,
trains it via **backpropagation through time (BPTT)** (Sections 8.2–8.3),
and applies it to character-level text generation on the Tiny Shakespeare dataset.
The forward pass follows Eqs. (8.3)–(8.4), and the backward pass derives gradients
as described in Section 8.3 (Eqs. 8.16–8.21).

### Notation

This notebook adopts the following conventions, consistent with the textbook:

| Symbol | Description |
|--------|-------------|
| $\mathbf{x}_t$ | Input vector at time step $t$ |
| $\mathbf{h}_t$ | Hidden state at time step $t$ |
| $\widehat{\mathbf{y}}_t$ | Predicted output (probability) at time step $t$ |
| $\mathbf{W}_{hx}$, $\mathbf{W}_{hh}$, $\mathbf{W}_{yh}$ | Weight matrices for input-to-hidden, hidden-to-hidden, hidden-to-output |
| $\mathbf{b}_h$, $\mathbf{b}_y$ | Bias vectors for hidden layer and output layer |
| $T$ | Sequence length (number of time steps) |
| $b$ | Batch size (number of independent sequences processed in parallel) |
| $n_x$, $n_h$, $n_y$ | Dimensions of input, hidden state, and output vectors |

The input tensor has shape $(b, T, n_x)$ in batch-major format (Section 8.5).

### Import libraries and dataset

In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import time

print('NumPy version:', np.__version__)
print('TensorFlow version:', tf.__version__)
print('Keras version:', keras.__version__)
print('TensorFlow Datasets version:', tfds.__version__)
print('Matplotlib version:', plt.matplotlib.__version__)

# Check available GPU devices
gpus = tf.config.list_physical_devices('GPU')
print("Num GPUs Available: ", len(gpus))

NumPy version: 2.0.2
TensorFlow version: 2.19.0
Keras version: 3.13.2
TensorFlow Datasets version: 4.9.9
Matplotlib version: 3.10.0
Num GPUs Available:  1


In [2]:
# Load Tiny Shakespeare dataset (public domain text)
(ds, ), ds_info = tfds.load(name='tiny_shakespeare', split=['train'], with_info=True)

ds = ds.map(lambda x: tf.strings.unicode_split(x['text'], 'UTF-8'))
for sample in ds.take(1):
    text_str = sample

print(ds_info)
print('Total number of characters:', len(text_str))

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/tiny_shakespeare/incomplete.8KUWDS_1.0.0/tiny_shakespeare-train.tfrecord*.…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/tiny_shakespeare/incomplete.8KUWDS_1.0.0/tiny_shakespeare-validation.tfrec…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/tiny_shakespeare/incomplete.8KUWDS_1.0.0/tiny_shakespeare-test.tfrecord*..…

Dataset tiny_shakespeare downloaded and prepared to /root/tensorflow_datasets/tiny_shakespeare/1.0.0. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='tiny_shakespeare',
    full_name='tiny_shakespeare/1.0.0',
    description="""
    40,000 lines of Shakespeare from a variety of Shakespeare's plays. Featured in
    Andrej Karpathy's blog post 'The Unreasonable Effectiveness of Recurrent Neural
    Networks': http://karpathy.github.io/2015/05/21/rnn-effectiveness/.
    
    To use for e.g. character modelling:
    
    ```
    d = tfds.load(name='tiny_shakespeare')['train']
    d = d.map(lambda x: tf.strings.unicode_split(x['text'], 'UTF-8'))
    # train split includes vocabulary for other splits
    vocabulary = sorted(set(next(iter(d)).numpy()))
    d = d.map(lambda x: {'cur_char': x[:-1], 'next_char': x[1:]})
    d = d.unbatch()
    seq_len = 100
    batch_size = 2
    d = d.batch(seq_len)
    d = d.batch(batch_size)
    ```
    """,
    homepage='https://githu

### Building vocabulary

Construct a character-level vocabulary that maps each unique character to an integer index.
This enables one-hot encoding of discrete tokens as input to the RNN (Section 8.7).

In [3]:
vocabulary = sorted(set(text_str.numpy()))

char_to_index = dict((char, index) for index, char in enumerate(vocabulary))

index_to_char = {}
for key, value in char_to_index.items():
    index_to_char[value] = key

print('Vocabulary size:', len(vocabulary))
print('Character to index mapping:\n', char_to_index)

Vocabulary size: 65
Character to index mapping:
 {b'\n': 0, b' ': 1, b'!': 2, b'$': 3, b'&': 4, b"'": 5, b',': 6, b'-': 7, b'.': 8, b'3': 9, b':': 10, b';': 11, b'?': 12, b'A': 13, b'B': 14, b'C': 15, b'D': 16, b'E': 17, b'F': 18, b'G': 19, b'H': 20, b'I': 21, b'J': 22, b'K': 23, b'L': 24, b'M': 25, b'N': 26, b'O': 27, b'P': 28, b'Q': 29, b'R': 30, b'S': 31, b'T': 32, b'U': 33, b'V': 34, b'W': 35, b'X': 36, b'Y': 37, b'Z': 38, b'a': 39, b'b': 40, b'c': 41, b'd': 42, b'e': 43, b'f': 44, b'g': 45, b'h': 46, b'i': 47, b'j': 48, b'k': 49, b'l': 50, b'm': 51, b'n': 52, b'o': 53, b'p': 54, b'q': 55, b'r': 56, b's': 57, b't': 58, b'u': 59, b'v': 60, b'w': 61, b'x': 62, b'y': 63, b'z': 64}


### Building training data pipeline

The text is split into fixed-length windows of size `seq_len`.
For each window, the input $\mathbf{x}_t$ is the current character and the target $\mathbf{y}_t$
is the next character, forming input–target pairs for the RNN.
This follows the sliding-window technique described in Section 8.9.

In [4]:
ids_from_chars = keras.layers.StringLookup(vocabulary=list(vocabulary), mask_token=None)
chars_from_ids = keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

text_ids = ids_from_chars(text_str)

n_voca = len(ids_from_chars.get_vocabulary())

seq_len = 128

# Split text into sequences of length (seq_len + 1)
ds_ids = tf.data.Dataset.from_tensor_slices(text_ids).batch(seq_len+1, drop_remainder=True)

# Create (input, target) pairs: input = chars[0:T], target = chars[1:T+1]
ds_xy = ds_ids.map(lambda x: {'cur_char': x[:-1], 'next_char': x[1:]})

buff_size = 10000
batch_size = 2
ds_train = ds_xy.shuffle(buff_size).batch(batch_size, drop_remainder=True)

ds_train

<_BatchDataset element_spec={'cur_char': TensorSpec(shape=(2, 128), dtype=tf.int64, name=None), 'next_char': TensorSpec(shape=(2, 128), dtype=tf.int64, name=None)}>

In [5]:
def prepare_dataset(sample, n_classes):
    """Convert integer-encoded characters to one-hot vectors."""
    X = keras.utils.to_categorical(sample['cur_char'], num_classes=n_classes)
    y = keras.utils.to_categorical(sample['next_char'], num_classes=n_classes)
    return X, y

def text_from_ids(ids):
    return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

# Verify the data pipeline
for sample in ds_train.take(1):
    X, y = prepare_dataset(sample, n_voca)
    X = np.argmax(X, axis=-1)
    y = np.argmax(y, axis=-1)
    print("Input :", text_from_ids(X[0]).numpy())
    print("Target:", text_from_ids(y[0]).numpy())

Input : b"s of the watery star hath been\nThe shepherd's note since we have left our throne\nWithout a burthen: time as long again\nWould be "
Target: b" of the watery star hath been\nThe shepherd's note since we have left our throne\nWithout a burthen: time as long again\nWould be f"


### Dimensions of variables

As described in Section 8.5, RNN inputs are organized as 3D tensors.

| Variable | Shape | Description |
|----------|-------|-------------|
| Input $\mathbf{x}$ | $(b, T, n_x)$ | Batch of one-hot encoded input sequences |
| Hidden state $\mathbf{h}$ | $(b, T, n_h)$ | Hidden states across all time steps |
| Output $\widehat{\mathbf{y}}$ | $(b, T, n_y)$ | Predicted probability distributions |
| Time slice $\mathbf{x}_t$ | $(b, n_x)$ | Input at a single time step $t$ |
| Time slice $\mathbf{h}_t$ | $(b, n_h)$ | Hidden state at time step $t$ |

At each time step, the RNN receives a 2D slice $(b, n_x)$ and produces
a hidden state of shape $(b, n_h)$, processing the batch in parallel
while maintaining sequential computation along the time axis (Section 8.5).

#### Check training data dimensions

In [6]:
for sample in ds_train.take(1):
    X, y = prepare_dataset(sample, n_voca)

print('Input  shape:', X.shape)  # (batch_size, seq_len, n_voca)
print('Output shape:', y.shape)  # (batch_size, seq_len, n_voca)

Input  shape: (2, 128, 66)
Output shape: (2, 128, 66)


### Exercise: Implement the vanilla RNN cell

**Forward pass** (Section 8.1, Eqs. 8.3–8.4):

1. Compute the hidden state: $\mathbf{h}_t = \tanh(\mathbf{W}_{hx}\mathbf{x}_t + \mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{b}_h)$ &ensp;(Eq. 8.3)
2. Compute the output: $\widehat{\mathbf{y}}_t = \text{softmax}(\mathbf{W}_{yh}\mathbf{h}_t + \mathbf{b}_y)$ &ensp;(Eq. 8.4)
3. Loop over all time steps $t = 1, \ldots, T$

**Backward pass** (Section 8.3, Eqs. 8.16–8.21):

At each time step $t$ (iterating backward from $T$ to $1$):

1. Accumulate output-layer gradients: $\partial\mathcal{L}/\partial\mathbf{W}_{yh}$, $\partial\mathcal{L}/\partial\mathbf{b}_y$ &ensp;(Eqs. 8.10–8.11)
2. Compute gradient w.r.t. $\mathbf{h}_t$ combining local loss and future contribution &ensp;(Eq. 8.17)
3. Backpropagate through $\tanh$: multiply by $(1 - \mathbf{h}_t^2)$ &ensp;(derivative of $\tanh$)
4. Accumulate recurrent-layer gradients: $\partial\mathcal{L}/\partial\mathbf{W}_{hx}$, $\partial\mathcal{L}/\partial\mathbf{W}_{hh}$, $\partial\mathcal{L}/\partial\mathbf{b}_h$ &ensp;(Eqs. 8.18–8.20)
5. Pass gradient to previous time step via $\mathbf{W}_{hh}$ &ensp;(Eq. 8.21)

**Note:** The gradient `dy` passed to the backward function is $\widehat{\mathbf{y}}_t - \mathbf{y}_t$ (Eq. 8.9),
which is the derivative of the cross-entropy loss with softmax activation.

### Helper functions

In [7]:
def softmax(arr):
    """Numerically stable softmax (Eq. 8.4)."""
    arr_shifted = arr - np.max(arr, axis=-1, keepdims=True)
    earr = np.exp(arr_shifted)
    return earr / np.sum(earr, axis=-1, keepdims=True)

def outer(a, b):
    """Batched outer product for gradient computation."""
    a = np.expand_dims(a, -1)
    b = np.expand_dims(b, -2)
    return np.matmul(a, b)

### RNN class definition

In [ ]:
class vanilla_rnn:

    def __init__(self, n_out, n_hidden, n_inp):
        # Hyperparameters
        self.n_inp = n_inp       # n_x: input dimension
        self.n_hidden = n_hidden # n_h: hidden state dimension
        self.n_out = n_out       # n_y: output dimension
        # Model parameters (Section 8.1)
        self.Whx = np.zeros((n_hidden, n_inp))    # W_hx: input-to-hidden weights
        self.Wyh = np.zeros((n_out, n_hidden))     # W_yh: hidden-to-output weights
        self.Whh = np.zeros((n_hidden, n_hidden))  # W_hh: hidden-to-hidden weights
        self.bh  = np.zeros((n_hidden, ))           # b_h: hidden bias
        self.by  = np.zeros((n_out, ))              # b_y: output bias
        # backup previous hidden state for BPTT
        self.hprev = np.zeros((1, n_hidden))        # h_{t-1}: initial hidden state

    def forward(self, xt, hprev, k1):
        """Forward pass through the RNN (Section 8.1, Eqs. 8.3-8.4)."""
        T = k1
        n_B, n_T, _ = xt.shape
        ht = np.zeros((n_B, n_T, self.n_hidden))  # (b, T, n_h)
        yhat = np.zeros((n_B, n_T, self.n_out))    # (b, T, n_y)
        self.hprev = np.copy(hprev)                   # Store initial hidden state for BPTT
        ht_prev = np.copy(hprev)                   # (b, n_h)

        for t in range(T):

            ### START CODE HERE ###

            # Compute hidden state: h_t = tanh(W_hx * x_t + W_hh * h_{t-1} + b_h)  (Eq. 8.3)
            ht[:,t] = None

            # Compute output: y_hat_t = softmax(W_yh * h_t + b_y)  (Eq. 8.4)
            yhat[:,t] = None

            # Carry hidden state forward to next time step
            ht_prev = None

            ### END CODE HERE ###

        return ht, yhat

    def backward(self, xt, ht, dy, k2):
        """Backward pass via BPTT (Section 8.3, Eqs. 8.16-8.21)."""
        T = k2
        n_B, _, _ = xt.shape
        dWhx = np.zeros((n_B, self.n_hidden, self.n_inp))
        dWhh = np.zeros((n_B, self.n_hidden, self.n_hidden))
        dWyh = np.zeros((n_B, self.n_out, self.n_hidden))
        dbh = np.zeros((n_B, self.n_hidden))
        dby = np.zeros((n_B, self.n_out))
        dhnext = np.zeros_like(ht[:,0])  # gradient from future time step

        for t in reversed(range(T)):

            ### START CODE HERE ###

            # Accumulate dW_yh: outer(dy_t, h_t)  (Eq. 8.10)
            dWyh += None

            # Accumulate db_y: dy_t  (Eq. 8.11)
            dby += None

            # Gradient w.r.t. h_t: combines local loss and future contribution  (Eq. 8.17)
            dht = None

            # Backpropagate through tanh: dz_t = (1 - h_t^2) * dh_t
            dtanh = None

            # Accumulate db_h: dz_t  (Eq. 8.20)
            dbh += None

            # Accumulate dW_hx: outer(dz_t, x_t)  (Eq. 8.18)
            dWhx += None

            # Accumulate dW_hh: outer(dz_t, h_{t-1})  (Eq. 8.19)
            ht_prev = None
            dWhh += None

            # Pass gradient to previous time step: W_hh^T * dz_t  (Eq. 8.21)
            dhnext = None

            ### END CODE HERE ###

        # Clip gradients to mitigate exploding gradients (Section 8.4)
        glist = [dWhx, dWhh, dWyh, dbh, dby]
        glist = [np.clip(dp, -10.0, 10.0) for dp in glist]
        dWhx, dWhh, dWyh, dbh, dby = glist

        # Average over the batch dimension
        dWhx = np.mean(dWhx, axis=0)
        dWhh = np.mean(dWhh, axis=0)
        dWyh = np.mean(dWyh, axis=0)
        dbh = np.mean(dbh, axis=0)
        dby = np.mean(dby, axis=0)

        return dWhx, dWhh, dWyh, dbh, dby

#### Test forward pass

In [9]:
np.random.seed(1)

rnn_cell = vanilla_rnn(2, 5, 3)

x_tmp = np.random.randn(10, 4, 3)    # batch=10, seq_len=4, n_x=3
h_prev_tmp = np.random.randn(10, 5)   # batch=10, n_h=5

rnn_cell.Whh = np.random.randn(5, 5)
rnn_cell.Whx = np.random.randn(5, 3)
rnn_cell.Wyh = np.random.randn(2, 5)
rnn_cell.bh = np.random.randn(5)
rnn_cell.by = np.random.randn(2)

h_next_tmp, yt_pred_tmp = rnn_cell.forward(x_tmp, h_prev_tmp, 4)

print("h_next[1,t,4] = \n", h_next_tmp[1,:,4])
print("h_next.shape = \n", h_next_tmp.shape)
print("yt_pred[3,t,1] =\n", yt_pred_tmp[3,:,1])
print("yt_pred.shape = \n", yt_pred_tmp.shape)

h_next[1,t,4] = 
 [ 0.99859767  0.72269806 -0.99831123 -0.99998484]
h_next.shape = 
 (10, 4, 5)
yt_pred[3,t,1] =
 [0.08039465 0.81391132 0.07988095 0.02803803]
yt_pred.shape = 
 (10, 4, 2)


**Expected output:**

```
h_next[1,t,4] =
 [ 0.99859767  0.72269806 -0.99831123 -0.99998484]
h_next.shape =
 (10, 4, 5)
yt_pred[3,t,1] =
 [0.08039465 0.81391132 0.07988095 0.02803803]
yt_pred.shape =
 (10, 4, 2)
```

#### Test backward pass

In [10]:
np.random.seed(1)

rnn_cell = vanilla_rnn(2, 5, 3)

x_tmp = np.random.randn(10, 4, 3)   # batch=10, seq_len=4, n_x=3
h0_tmp = np.random.randn(10, 5)      # batch=10, n_h=5

rnn_cell.Whx = np.random.randn(5, 3)
rnn_cell.Whh = np.random.randn(5, 5)
rnn_cell.Wyh = np.random.randn(2, 5)
rnn_cell.bh = np.random.randn(5)
rnn_cell.by = np.random.randn(2)

h_tmp, y_tmp = rnn_cell.forward(x_tmp, h0_tmp, 4)

dy_tmp = np.random.randn(10, 4, 2)

# Note: backward now requires hprev (initial hidden state)
dWhx, dWhh, dWyh, dbh, dby = rnn_cell.backward(x_tmp, h_tmp, dy_tmp, 4)

print("dWhx[3][1] =", dWhx[3, 1])
print("dWhx.shape =", dWhx.shape)
print("dWhh[1][2] =", dWhh[1, 2])
print("dWhh.shape =", dWhh.shape)
print("dbh[4] =", dbh[4])
print("dbh.shape =", dbh.shape)
print("dby[1] =", dby[1])
print("dby.shape =", dby.shape)

dWhx[3][1] = -1.1757864340551145
dWhx.shape = (5, 3)
dWhh[1][2] = -0.4826603984266227
dWhh.shape = (5, 5)
dbh[4] = -0.7891177327315932
dbh.shape = (5,)
dby[1] = -0.24507440509183692
dby.shape = (2,)


**Expected output:**
```
dWhx[3][1] = -1.1757864340551145
dWhx.shape = (5, 3)
dWhh[1][2] = -0.4826603984266227
dWhh.shape = (5, 5)
dbh[4] = -0.7891177327315932
dbh.shape = (5,)
dby[1] = -0.24507440509183692
dby.shape = (2,)
```

In [11]:
class embedding():
    """Placeholder for an embedding layer (Section 8.7)."""
    def __init__(self, n_vocab, n_embedding):
        self.n_vocab = n_vocab
        self.n_embedding = n_embedding
        self.embedding = np.zeros((n_vocab, n_embedding))

    def forward(self):
        return

### Create RNN and initialize weights

The input-to-hidden and hidden-to-output weights are initialized with uniform random values
scaled by $1/\sqrt{n}$. The hidden-to-hidden weight matrix $\mathbf{W}_{hh}$ is initialized
using **QR decomposition** to obtain an orthogonal matrix, following the orthogonal
initialization strategy described in Section 8.4 (Eqs. 8.22–8.23).

In [12]:
# Parameters defined earlier:
#   n_voca   = vocabulary size (66)
#   seq_len  = 128 (sequence length)
#   batch_size = 2

n_hidden = 256

RNN = vanilla_rnn(n_out=n_voca, n_hidden=n_hidden, n_inp=n_voca)

# Xavier-style uniform initialization for input and output weights
RNN.Whx = np.random.uniform(-np.sqrt(1./n_voca), np.sqrt(1./n_voca),
                             (n_hidden, n_voca))
RNN.Wyh = np.random.uniform(-np.sqrt(1./n_hidden), np.sqrt(1./n_hidden),
                             (n_voca, n_hidden))

# Orthogonal initialization for recurrent weights (Section 8.4)
RNN.Whh, _ = np.linalg.qr(
    np.random.uniform(-np.sqrt(1./n_hidden), np.sqrt(1./n_hidden),
                       (n_hidden, n_hidden)))

### Test RNN before training

Before training, the randomly initialized RNN generates meaningless character sequences.

In [13]:
X_test = np.zeros((1, 1, n_voca), dtype=float)
ix = np.random.randint(n_voca)
X_test[0, 0, ix] = 1.0
y_char = index_to_char[ix]

print('The RNN is initiated with', y_char, 'and will generate 100 characters.')

result = ''
t_length = 100
hprev = np.random.randn(1, n_hidden)

for i in range(t_length):
    ht, y_pred = RNN.forward(X_test, hprev, k1=1)
    hprev = ht[:, -1]

    iy = np.argmax(y_pred, -1)
    X_test = np.zeros((1, 1, n_voca))
    X_test[0, 0, iy[0, 0]] = 1.0

    pred_ids = np.squeeze(iy, axis=-1)
    result += text_from_ids(pred_ids).numpy().decode('utf-8')

print('Generated text:\n', result)

The RNN is initiated with b'3' and will generate 100 characters.
Generated text:
 fL&3:bTTv[UNK]kkz;n-R.;[UNK]$HSrOusBxh&XPt.BkzPE-P,G
LbpfccIeKrgB!t;UJ3egf[UNK]g[UNK]IdhKIkWk!mVU;dXZ:Avk b'U!tuXJsA


In [14]:
def update_param(RNN, dWhx, dWhh, dWyh, dbh, dby, alpha=1e-3):
    """Update parameters using vanilla SGD with learning rate alpha."""
    RNN.Whx -= alpha * dWhx
    RNN.Whh -= alpha * dWhh
    RNN.Wyh -= alpha * dWyh
    RNN.bh  -= alpha * dbh
    RNN.by  -= alpha * dby
    return

### Training

The training loop implements BPTT (Section 8.2) with the full sequence length.
The cross-entropy loss $\mathcal{L} = -\frac{1}{N}\sum_t \sum_k y_{t,k} \log \widehat{y}_{t,k}$ (Eq. 8.6)
is computed at each step, and the gradient $\widehat{\mathbf{y}}_t - \mathbf{y}_t$ (Eq. 8.9)
is passed to the backward function.

**Note:** RNN training takes considerably more time than MLP or CNN training
due to the sequential nature of computation along the time dimension.

In [15]:
import tqdm

steps = len(ds_train)
print(steps)

3890


In [16]:
n_epochs = 5

# Initialize hidden state (can be zeros or random)
hprev = np.random.randn(batch_size, n_hidden)

for epoch in range(n_epochs):
    sample_no = 0
    t_loss = 0.0
    start = time.time()

    pbar = tqdm.notebook.tqdm(ds_train, total=steps)
    pbar.set_description('Epoch:%2d' % (epoch+1))

    for sample in pbar:

        X_train, y_train = prepare_dataset(sample, n_voca)

        # Forward pass (Eqs. 8.3-8.4)
        ht, y_pred = RNN.forward(X_train, hprev, k1=seq_len)

        # Carry hidden state to next batch (detach from computation graph)
        hprev = ht[:, -1]

        # Gradient of cross-entropy + softmax: dy = y_hat - y  (Eq. 8.9)
        dy = y_pred - y_train

        # Backward pass via BPTT (Eqs. 8.16-8.21)
        dWhx, dWhh, dWyh, dbh, dby = RNN.backward(
            X_train, ht, dy, k2=seq_len)

        # Parameter update (vanilla SGD)
        update_param(RNN, dWhx, dWhh, dWyh, dbh, dby, alpha=1e-3)

        sample_no += 1

        # Cross-entropy loss (Eq. 8.6)
        loss_J = -np.mean(np.sum(y_train * np.log(y_pred), axis=-1))
        pbar.set_postfix({'loss': loss_J})
        t_loss += loss_J

    print('Epoch:%2d/ Samples:%4d, Elapsed_t: %4.2fs,  loss: %10.8f'
          % (epoch+1, sample_no, time.time() - start, t_loss/steps))

  0%|          | 0/3890 [00:00<?, ?it/s]

Epoch: 1/ Samples:3890, Elapsed_t: 1946.80s,  loss: 2.47034121


  0%|          | 0/3890 [00:00<?, ?it/s]

Epoch: 2/ Samples:3890, Elapsed_t: 1939.89s,  loss: 2.12503767


  0%|          | 0/3890 [00:00<?, ?it/s]

Epoch: 3/ Samples:3890, Elapsed_t: 1880.05s,  loss: 1.99360561


  0%|          | 0/3890 [00:00<?, ?it/s]

Epoch: 4/ Samples:3890, Elapsed_t: 1876.69s,  loss: 1.90533638


  0%|          | 0/3890 [00:00<?, ?it/s]

Epoch: 5/ Samples:3890, Elapsed_t: 1896.55s,  loss: 1.84390974


### Test model with a random character

After training, the model generates text that resembles character-level patterns
in the training data, though the output generally lacks grammatical correctness.

In [17]:
X_test = np.zeros((1, 1, n_voca), dtype=float)
ix = np.random.randint(n_voca)
X_test[0, 0, ix] = 1.0
y_char = index_to_char[ix]

print('The RNN is initiated with', y_char, 'and will generate 100 characters.')

result = ''
t_length = 100
hprev = np.random.randn(1, n_hidden)

for i in range(t_length):
    ht, y_pred = RNN.forward(X_test, hprev, k1=1)
    hprev = ht[:, -1]

    iy = np.argmax(y_pred, -1)
    X_test = np.zeros((1, 1, n_voca))
    X_test[0, 0, iy[0, 0]] = 1.0

    pred_ids = np.squeeze(iy, axis=-1)
    result += text_from_ids(pred_ids).numpy().decode('utf-8')

print('Generated text:\n', result)

The RNN is initiated with b'H' and will generate 100 characters.
Generated text:
 d:
That hath have been the caments and the day the wasting and the day the wasting and the day the w


### Save and load model

In [18]:
import pickle

def save_object(obj):
    try:
        with open("RNN.pickle", "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    except Exception as ex:
        print("Error during pickling object:", ex)

def load_object(filename):
    try:
        with open(filename, "rb") as f:
            return pickle.load(f)
    except Exception as ex:
        print("Error during unpickling object:", ex)

In [19]:
save_object(RNN)

In [20]:
RNN = load_object("RNN.pickle")

(c) 2026 S. W. Lee